# Pandas Module: Creating and Updating Columns from Other Columns

## Beginner-friendly comprehensive module

This notebook teaches a very common real-world data task:

> How do we create, update, delete, and encode columns based on other columns?

We will practice:

- Creating new columns from one column
- Creating new columns from multiple columns
- Updating column values with `.loc`
- Creating columns using conditions
- Creating columns using functions
- Using `apply()` and `lambda`
- Creating columns based on subgroup logic using `groupby().transform()`
- Creating dummy variables
- Deleting columns
- Avoiding common beginner mistakes

The examples use transaction and customer data.

In [ ]:
import pandas as pd
import numpy as np

## 1. Load the data

We have two files:

- `transactions_advanced.csv`: transaction-level data
- `customers_advanced.csv`: customer-level data

A transaction table usually has one row per order.
A customer table usually has one row per customer.

In [ ]:
transactions = pd.read_csv("transactions_advanced.csv")
customers = pd.read_csv("customers_advanced.csv")

transactions.head()

In [ ]:
customers.head()

## 2. Inspect the data

Before creating or changing columns, always inspect the data.

Useful checks:

- What columns do we have?
- What are the data types?
- Are there missing values?
- What do the first few rows look like?

In [ ]:
transactions.info()

In [ ]:
transactions.describe(include="all")

In [ ]:
transactions.isna().sum()

## 3. Create a column using simple arithmetic

A very common task is creating a new column from existing numeric columns.

Example business question:

> What is the net sales amount for each transaction?

Here we define:

`net_sales = gross_sales - discount_amount + shipping_fee`

Important business note:

- Add `shipping_fee` if it is money charged to the customer and counted as revenue.
- Subtract shipping if it is a shipping cost paid by the company.
- In this teaching dataset, `shipping_fee` means the fee charged to the customer.

In [ ]:
transactions["net_sales"] = (
    transactions["gross_sales"]
    - transactions["discount_amount"]
    + transactions["shipping_fee"]
)

transactions[[
    "gross_sales", 
    "discount_amount", 
    "shipping_fee", 
    "net_sales"
]].head()

### Exercise 1

Create a new column called `discount_rate`.

Formula:

`discount_rate = discount_amount / gross_sales`

Then show the first 5 rows of:

- `gross_sales`
- `discount_amount`
- `discount_rate`

In [ ]:
# Exercise 1: Your code here

### Solution 1

In [ ]:
transactions["discount_rate"] = (
    transactions["discount_amount"] / transactions["gross_sales"]
)

transactions[[
    "gross_sales", 
    "discount_amount", 
    "discount_rate"
]].head()

## 4. Create a column using a condition

Sometimes we want a new column that flags whether something is true.

Example:

> Is this a high-value order?

We can create a column that is:

- `True` if `net_sales > 100`
- `False` otherwise

In [ ]:
transactions["high_value_order"] = transactions["net_sales"] > 100

transactions[[
    "net_sales", 
    "high_value_order"
]].head()

If we want 1 and 0 instead of True and False, we can use `.astype(int)`.

In [ ]:
transactions["high_value_order_int"] = (
    transactions["net_sales"] > 100
).astype(int)

transactions[[
    "net_sales", 
    "high_value_order_int"
]].head()

### Exercise 2

Create a column called `has_discount`.

It should be:

- `1` if `discount_amount > 0`
- `0` otherwise

In [ ]:
# Exercise 2: Your code here

### Solution 2

In [ ]:
transactions["has_discount"] = (
    transactions["discount_amount"] > 0
).astype(int)

transactions[[
    "discount_amount", 
    "has_discount"
]].head()

## 5. Update values with `.loc`

`.loc` is one of the most important pandas tools.

The pattern is:

```python
df.loc[condition, "column_name"] = new_value
```

Read it as:

> In rows where the condition is true, update this column.

This avoids the common chained-assignment problem.

In [ ]:
# Start with a default value
transactions["sales_level"] = "Regular"

# Update only rows where net_sales is greater than 150
transactions.loc[
    transactions["net_sales"] > 150,
    "sales_level"
] = "High"

transactions[[
    "net_sales", 
    "sales_level"
]].head(10)

### Important beginner rule

Do **not** do this:

```python
transactions[transactions["net_sales"] > 150]["sales_level"] = "High"
```

That may create a copy and not update the original DataFrame.

Use `.loc` instead.

### Exercise 3

Create a column called `return_status`.

Steps:

1. Set all values to `"Kept"`
2. Use `.loc` to set it to `"Returned"` when `is_return == 1`

In [ ]:
# Exercise 3: Your code here

### Solution 3

In [ ]:
transactions["return_status"] = "Kept"

transactions.loc[
    transactions["is_return"] == 1,
    "return_status"
] = "Returned"

transactions[[
    "is_return", 
    "return_status"
]].head(10)

## 6. Multiple conditions with `.loc`

When we have multiple conditions:

- use `&` for AND
- use `|` for OR
- wrap each condition in parentheses

Example:

> Flag orders that are both high-value and returned.

In [ ]:
transactions["large_return"] = 0

transactions.loc[
    (transactions["net_sales"] > 100) & (transactions["is_return"] == 1),
    "large_return"
] = 1

transactions[[
    "net_sales", 
    "is_return", 
    "large_return"
]].head(10)

### Exercise 4

Create a column called `priority_review`.

It should be `1` when:

- `net_sales > 150`, OR
- `is_return == 1`

Otherwise it should be `0`.

In [ ]:
# Exercise 4: Your code here

### Solution 4

In [ ]:
transactions["priority_review"] = 0

transactions.loc[
    (transactions["net_sales"] > 150) | (transactions["is_return"] == 1),
    "priority_review"
] = 1

transactions[[
    "net_sales", 
    "is_return", 
    "priority_review"
]].head(10)

## 7. Create categories with `np.select`

When there are several conditions, repeated `.loc` works, but `np.select()` is cleaner.

Example:

> Create an order size category.

Rules:

- `Large` if `net_sales >= 150`
- `Medium` if `net_sales >= 75`
- `Small` otherwise

In [ ]:
conditions = [
    transactions["net_sales"] >= 150,
    transactions["net_sales"] >= 75
]

choices = [
    "Large",
    "Medium"
]

transactions["order_size"] = np.select(
    conditions,
    choices,
    default="Small"
)

transactions[[
    "net_sales", 
    "order_size"
]].head(10)

### Exercise 5

Create a column called `discount_level`.

Rules:

- `"No Discount"` if `discount_rate == 0`
- `"Small Discount"` if `discount_rate` is greater than 0 and less than 0.10
- `"Large Discount"` if `discount_rate >= 0.10`

In [ ]:
# Exercise 5: Your code here

### Solution 5

In [ ]:
conditions = [
    transactions["discount_rate"] == 0,
    (transactions["discount_rate"] > 0) & (transactions["discount_rate"] < 0.10),
    transactions["discount_rate"] >= 0.10
]

choices = [
    "No Discount",
    "Small Discount",
    "Large Discount"
]

transactions["discount_level"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

transactions[[
    "discount_rate", 
    "discount_level"
]].head(10)

## 8. Create a column with `.assign()`

`.assign()` creates new columns and is useful when writing a clean pipeline.

This:

```python
df = df.assign(new_column = ...)
```

means:

> Return a DataFrame with a new column.

Inside `.assign()`, we often use `lambda x:`.

Here, `x` means the current DataFrame at that point in the pipeline.

In [ ]:
transactions = transactions.assign(
    revenue_after_return=lambda x: np.where(
        x["is_return"] == 1,
        0,
        x["net_sales"]
    )
)

transactions[[
    "net_sales", 
    "is_return", 
    "revenue_after_return"
]].head(10)

### Exercise 6

Use `.assign()` to create a column called `sales_after_discount`.

Formula:

`gross_sales - discount_amount`

In [ ]:
# Exercise 6: Your code here

### Solution 6

In [ ]:
transactions = transactions.assign(
    sales_after_discount=lambda x: x["gross_sales"] - x["discount_amount"]
)

transactions[[
    "gross_sales", 
    "discount_amount", 
    "sales_after_discount"
]].head()

## 9. Use a custom function for column creation

Sometimes the rule is too complex to write neatly in one line.

Then we can define a function.

Example:

> Create a customer-facing order label.

In [ ]:
def label_order(row):
    """Return a simple label for each order."""

    if row["is_return"] == 1:
        return "Returned Order"
    elif row["net_sales"] >= 150:
        return "High Value Order"
    elif row["discount_amount"] > 0:
        return "Discounted Order"
    else:
        return "Regular Order"

transactions["order_label"] = transactions.apply(label_order, axis=1)

transactions[[
    "net_sales", 
    "discount_amount", 
    "is_return", 
    "order_label"
]].head(10)

### What does `axis=1` mean?

`axis=1` means:

> Apply the function row by row.

Inside the function, `row["net_sales"]` refers to the value of `net_sales` in that row.

### Beginner advice

Use `apply(axis=1)` only when the logic is hard to express with vectorized pandas operations.

For simple math and conditions, prefer direct pandas operations or `.loc`.

### Exercise 7

Create a function called `shipping_label(row)`.

Rules:

- If `shipping_fee == 0`, return `"Free Shipping"`
- If `shipping_fee > 0` and `net_sales >= 100`, return `"Paid Shipping - High Value"`
- Otherwise return `"Paid Shipping"`

Then create a new column called `shipping_label`.

In [ ]:
# Exercise 7: Your code here

### Solution 7

In [ ]:
def shipping_label(row):
    if row["shipping_fee"] == 0:
        return "Free Shipping"
    elif (row["shipping_fee"] > 0) and (row["net_sales"] >= 100):
        return "Paid Shipping - High Value"
    else:
        return "Paid Shipping"

transactions["shipping_label"] = transactions.apply(
    shipping_label,
    axis=1
)

transactions[[
    "shipping_fee", 
    "net_sales", 
    "shipping_label"
]].head(10)

## 10. Group-based column creation with `groupby().transform()`

This is one of the most useful advanced beginner skills.

Sometimes we need to create a column based on the subgroup that each row belongs to.

Examples:

- Compare each order to the average order in the same country
- Compute each order's share of category sales
- Rank orders within each customer
- Flag top orders within each channel

The key tool is:

```python
groupby(...).transform(...)
```

Why `transform()`?

Because it returns one value for each original row, so we can assign it back as a new column.

### Example 1: Country average sales for each row

Question:

> What is the average net sales in each country?

Then attach that country average back to every transaction.

In [ ]:
transactions["country_avg_sales"] = (
    transactions
    .groupby("country")["net_sales"]
    .transform("mean")
)

transactions[[
    "country", 
    "net_sales", 
    "country_avg_sales"
]].head(10)

Now we can compare each transaction to its own country average.

In [ ]:
transactions["above_country_average"] = (
    transactions["net_sales"] > transactions["country_avg_sales"]
).astype(int)

transactions[[
    "country", 
    "net_sales", 
    "country_avg_sales", 
    "above_country_average"
]].head(10)

### Exercise 8

Create a column called `category_avg_sales`.

It should contain the average `net_sales` for that row's `category`.

Then create `above_category_average`, equal to `1` if the row's `net_sales` is greater than its category average.

In [ ]:
# Exercise 8: Your code here

### Solution 8

In [ ]:
transactions["category_avg_sales"] = (
    transactions
    .groupby("category")["net_sales"]
    .transform("mean")
)

transactions["above_category_average"] = (
    transactions["net_sales"] > transactions["category_avg_sales"]
).astype(int)

transactions[[
    "category", 
    "net_sales", 
    "category_avg_sales", 
    "above_category_average"
]].head(10)

## 11. Percentage share within a subgroup

Business question:

> What percentage of country sales does each transaction represent?

Steps:

1. Compute total sales by country
2. Attach the country total to each row
3. Divide each row's sales by its country total

In [ ]:
transactions["country_total_sales"] = (
    transactions
    .groupby("country")["net_sales"]
    .transform("sum")
)

transactions["share_of_country_sales"] = (
    transactions["net_sales"] / transactions["country_total_sales"]
)

transactions[[
    "country", 
    "net_sales", 
    "country_total_sales", 
    "share_of_country_sales"
]].head(10)

### Exercise 9

Create a column called `share_of_category_sales`.

It should equal:

`net_sales / total net_sales in that category`

In [ ]:
# Exercise 9: Your code here

### Solution 9

In [ ]:
transactions["category_total_sales"] = (
    transactions
    .groupby("category")["net_sales"]
    .transform("sum")
)

transactions["share_of_category_sales"] = (
    transactions["net_sales"] / transactions["category_total_sales"]
)

transactions[[
    "category", 
    "net_sales", 
    "category_total_sales", 
    "share_of_category_sales"
]].head(10)

## 12. Rank rows within a subgroup

Business question:

> Within each country, which orders are largest?

We can use `rank()` inside `groupby()`.

In [ ]:
transactions["rank_in_country"] = (
    transactions
    .groupby("country")["net_sales"]
    .rank(ascending=False, method="dense")
)

transactions[[
    "country", 
    "net_sales", 
    "rank_in_country"
]].sort_values(["country", "rank_in_country"]).head(15)

### Exercise 10

Create a column called `rank_in_category`.

Rank orders within each category by `net_sales`, from largest to smallest.

In [ ]:
# Exercise 10: Your code here

### Solution 10

In [ ]:
transactions["rank_in_category"] = (
    transactions
    .groupby("category")["net_sales"]
    .rank(ascending=False, method="dense")
)

transactions[[
    "category", 
    "net_sales", 
    "rank_in_category"
]].sort_values(["category", "rank_in_category"]).head(15)

## 13. Flag top rows within each subgroup

Business question:

> Is this order in the top 10% of sales within its country?

We can calculate the 90th percentile within each group.

In [ ]:
country_90th = (
    transactions
    .groupby("country")["net_sales"]
    .transform(lambda x: x.quantile(0.90))
)

transactions["top_10pct_in_country"] = (
    transactions["net_sales"] >= country_90th
).astype(int)

transactions[[
    "country", 
    "net_sales", 
    "top_10pct_in_country"
]].head(10)

### Exercise 11

Create a column called `top_25pct_in_channel`.

It should be `1` if an order is in the top 25% of `net_sales` within its `channel`.

Hint: top 25% means values greater than or equal to the 75th percentile.

In [ ]:
# Exercise 11: Your code here

### Solution 11

In [ ]:
channel_75th = (
    transactions
    .groupby("channel")["net_sales"]
    .transform(lambda x: x.quantile(0.75))
)

transactions["top_25pct_in_channel"] = (
    transactions["net_sales"] >= channel_75th
).astype(int)

transactions[[
    "channel", 
    "net_sales", 
    "top_25pct_in_channel"
]].head(10)

## 14. Group-based cumulative columns

Sometimes we want running totals.

Business question:

> For each customer, what is the running total of net sales over time?

Before using cumulative sums, sort the data.

In [ ]:
transactions["order_date"] = pd.to_datetime(transactions["order_date"])

transactions = transactions.sort_values([
    "customer_id", 
    "order_date"
])

transactions["customer_running_sales"] = (
    transactions
    .groupby("customer_id")["net_sales"]
    .cumsum()
)

transactions[[
    "customer_id", 
    "order_date", 
    "net_sales", 
    "customer_running_sales"
]].head(15)

### Exercise 12

Create a column called `customer_order_number`.

It should be:

- 1 for the customer's first order
- 2 for the customer's second order
- 3 for the customer's third order
- and so on

Hint: use `groupby().cumcount()`.

In [ ]:
# Exercise 12: Your code here

### Solution 12

In [ ]:
transactions["customer_order_number"] = (
    transactions
    .groupby("customer_id")
    .cumcount()
    + 1
)

transactions[[
    "customer_id", 
    "order_date", 
    "order_id", 
    "customer_order_number"
]].head(15)

## 15. Create columns after merging tables

Often, the columns we need are in different tables.

Let's merge transactions with customer data.

In [ ]:
tx_all = transactions.merge(
    customers,
    on="customer_id",
    how="left",
    indicator=True
)

tx_all.head()

### What is `_merge`?

Because we used `indicator=True`, pandas created a column called `_merge`.

It tells us whether each row matched:

- `both`: found in both tables
- `left_only`: found only in transactions
- `right_only`: found only in customers

For a left join, `left_only` means a transaction did not find a matching customer.

In [ ]:
tx_all["_merge"].value_counts()

In [ ]:
# After checking merge quality, we can remove the helper column.
tx_all = tx_all.drop(columns=["_merge"])

## 16. Create columns using transaction + customer columns

Now we can create columns using both transaction information and customer information.

Example:

> Is this a loyal customer's high-value purchase?

In [ ]:
tx_all["loyal_high_value"] = (
    (tx_all["loyalty_tier"].isin(["Gold", "Platinum"]))
    & (tx_all["net_sales"] > 100)
).astype(int)

tx_all[[
    "customer_id", 
    "loyalty_tier", 
    "net_sales", 
    "loyal_high_value"
]].head(10)

### Exercise 13

Create a column called `student_discount_flag`.

It should be `1` when:

- `segment` is `"Student"`
- and `discount_amount > 0`

Otherwise it should be `0`.

In [ ]:
# Exercise 13: Your code here

### Solution 13

In [ ]:
tx_all["student_discount_flag"] = (
    (tx_all["segment"] == "Student")
    & (tx_all["discount_amount"] > 0)
).astype(int)

tx_all[[
    "segment", 
    "discount_amount", 
    "student_discount_flag"
]].head(10)

## 17. Create dummy columns

Dummy columns convert categories into 0/1 columns.

This is useful for:

- machine learning
- regression
- simple reporting
- converting text categories into numeric features

In [ ]:
channel_dummies = pd.get_dummies(
    tx_all["channel"],
    prefix="channel",
    dtype=int
)

channel_dummies.head()

In [ ]:
tx_all_with_dummies = pd.concat(
    [tx_all, channel_dummies],
    axis=1
)

tx_all_with_dummies.head()

### Exercise 14

Create dummy columns for `loyalty_tier`.

Use prefix `"tier"`.

Then combine the dummy columns with `tx_all`.

In [ ]:
# Exercise 14: Your code here

### Solution 14

In [ ]:
tier_dummies = pd.get_dummies(
    tx_all["loyalty_tier"],
    prefix="tier",
    dtype=int
)

tx_all = pd.concat(
    [tx_all, tier_dummies],
    axis=1
)

tx_all.head()

## 18. Delete columns

Sometimes we create temporary helper columns and then remove them.

Use:

```python
df.drop(columns=["column_name"])
```

By default, this returns a new DataFrame.

In [ ]:
# Example: create a temporary column
tx_all["temporary_column"] = tx_all["net_sales"] * 2

# Remove the temporary column
tx_all = tx_all.drop(columns=["temporary_column"])

tx_all.head()

### Exercise 15

Drop the columns `country_avg_sales` and `country_total_sales` from the `transactions` DataFrame.

Use `errors="ignore"` so the code will not break if the columns are already gone.

In [ ]:
# Exercise 15: Your code here

### Solution 15

In [ ]:
transactions = transactions.drop(
    columns=["country_avg_sales", "country_total_sales"],
    errors="ignore"
)

transactions.head()

## 19. Mini case study: Executive country dashboard

Now we combine many ideas:

- groupby
- aggregation
- creating new columns with `.assign()`
- sorting

Business question:

> For each country, summarize performance for executives.

In [ ]:
executive_country = (
    tx_all
    .groupby("country")
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean"),
        avg_marketing_spend=("marketing_spend", "mean")
    )
    .assign(
        sales_per_customer=lambda x: x["total_sales"] / x["unique_customers"]
    )
    .sort_values("total_sales", ascending=False)
)

executive_country

### What does `.assign()` do here?

`.assign()` adds a new column to the summary table.

Here:

```python
sales_per_customer = total_sales / unique_customers
```

The `lambda x:` means:

> Use the DataFrame created by the previous step.

### Exercise 16

Create a category-level executive summary.

For each `category`, calculate:

- `total_sales`
- `transactions`
- `avg_sales`
- `return_rate`

Then use `.assign()` to create:

`sales_per_transaction = total_sales / transactions`

Sort by `total_sales` from largest to smallest.

In [ ]:
# Exercise 16: Your code here

### Solution 16

In [ ]:
executive_category = (
    tx_all
    .groupby("category")
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean")
    )
    .assign(
        sales_per_transaction=lambda x: x["total_sales"] / x["transactions"]
    )
    .sort_values("total_sales", ascending=False)
)

executive_category

## 20. Common beginner mistakes

### Mistake 1: Chained assignment

Avoid:

```python
df[df["net_sales"] > 100]["flag"] = 1
```

Use:

```python
df.loc[df["net_sales"] > 100, "flag"] = 1
```

### Mistake 2: Forgetting parentheses around conditions

Wrong:

```python
df["net_sales"] > 100 & df["is_return"] == 1
```

Correct:

```python
(df["net_sales"] > 100) & (df["is_return"] == 1)
```

### Mistake 3: Using `apply()` for everything

`apply()` is useful, but often slower and harder to read.

Prefer direct pandas operations when possible.

## 21. Final practice set

Try these without looking at the solutions first.

### Final Exercise A

Create a column called `profitable_order`.

It should be `1` if:

`net_sales > marketing_spend`

Otherwise `0`.

In [ ]:
# Final Exercise A: Your code here

### Solution A

In [ ]:
tx_all["profitable_order"] = (
    tx_all["net_sales"] > tx_all["marketing_spend"]
).astype(int)

tx_all[[
    "net_sales", 
    "marketing_spend", 
    "profitable_order"
]].head()

### Final Exercise B

Create a column called `channel_avg_sales`.

It should contain the average `net_sales` within each `channel`.

Then create `above_channel_avg`, equal to 1 if the order is above its channel average.

In [ ]:
# Final Exercise B: Your code here

### Solution B

In [ ]:
tx_all["channel_avg_sales"] = (
    tx_all
    .groupby("channel")["net_sales"]
    .transform("mean")
)

tx_all["above_channel_avg"] = (
    tx_all["net_sales"] > tx_all["channel_avg_sales"]
).astype(int)

tx_all[[
    "channel", 
    "net_sales", 
    "channel_avg_sales", 
    "above_channel_avg"
]].head()

### Final Exercise C

Create a column called `customer_total_sales`.

It should contain the total sales by each customer.

Then create `share_of_customer_sales`.

Formula:

`net_sales / customer_total_sales`

In [ ]:
# Final Exercise C: Your code here

### Solution C

In [ ]:
tx_all["customer_total_sales"] = (
    tx_all
    .groupby("customer_id")["net_sales"]
    .transform("sum")
)

tx_all["share_of_customer_sales"] = (
    tx_all["net_sales"] / tx_all["customer_total_sales"]
)

tx_all[[
    "customer_id", 
    "net_sales", 
    "customer_total_sales", 
    "share_of_customer_sales"
]].head()

### Final Exercise D

Create a column called `customer_first_order`.

It should be `1` if this is the customer's first order, otherwise `0`.

Hint:

- Sort by `customer_id` and `order_date`
- Use `groupby().cumcount()`

In [ ]:
# Final Exercise D: Your code here

### Solution D

In [ ]:
tx_all = tx_all.sort_values([
    "customer_id", 
    "order_date"
])

tx_all["customer_order_number"] = (
    tx_all
    .groupby("customer_id")
    .cumcount()
    + 1
)

tx_all["customer_first_order"] = (
    tx_all["customer_order_number"] == 1
).astype(int)

tx_all[[
    "customer_id", 
    "order_date", 
    "customer_order_number", 
    "customer_first_order"
]].head(15)

## 22. Summary

In this module, you learned how to create and update columns using:

| Method | Use case |
|---|---|
| Direct arithmetic | Create columns from numeric columns |
| Boolean conditions | Create True/False or 0/1 flags |
| `.loc` | Safely update rows that meet a condition |
| `np.select()` | Create categories from multiple conditions |
| `.assign()` | Add columns inside a pipeline |
| custom functions + `.apply()` | Use complex row-level logic |
| `groupby().transform()` | Create row-level columns based on subgroup values |
| `rank()` and `cumsum()` | Create within-group rankings and running totals |
| `pd.get_dummies()` | Convert categories into dummy columns |
| `.drop()` | Delete unnecessary columns |

Most important takeaway:

> Creating good columns is one of the most important skills in data analysis.

# Additional Practice: More Exercises on Column Creation

This expanded section gives extra classroom practice for each major concept:

- direct arithmetic
- `.loc`
- `np.where`
- `np.select`
- `lambda`
- `.apply()`
- custom `def` functions
- `.assign()`
- group-based columns with `groupby().transform()`
- dummy variables
- deleting and renaming columns

Each exercise includes a solution.

## A. More Practice: Direct Column Creation

These are the simplest column-creation exercises.  
They help students practice building columns from existing numeric columns.

### Exercise A1: Create `order_total_before_return`

Create a column:

`order_total_before_return = gross_sales - discount_amount + shipping_fee`

In [ ]:
# Exercise A1: Your code here

### Solution A1

In [ ]:
tx_all["order_total_before_return"] = (
    tx_all["gross_sales"]
    - tx_all["discount_amount"]
    + tx_all["shipping_fee"]
)

tx_all[[
    "gross_sales",
    "discount_amount",
    "shipping_fee",
    "order_total_before_return"
]].head()

### Exercise A2: Create `discount_percent`

Create a column:

`discount_percent = discount_rate * 100`

In [ ]:
# Exercise A2: Your code here

### Solution A2

In [ ]:
tx_all["discount_percent"] = tx_all["discount_rate"] * 100

tx_all[[
    "discount_rate",
    "discount_percent"
]].head()

### Exercise A3: Create `shipping_share`

Create a column:

`shipping_share = shipping_fee / order_total_before_return`

This tells us how much of the order total comes from shipping fees.

In [ ]:
# Exercise A3: Your code here

### Solution A3

In [ ]:
tx_all["shipping_share"] = (
    tx_all["shipping_fee"] / tx_all["order_total_before_return"]
)

tx_all[[
    "shipping_fee",
    "order_total_before_return",
    "shipping_share"
]].head()

## B. More Practice: `.loc`

`.loc` is best when we want to update values in selected rows.

Pattern:

```python
df.loc[condition, "column"] = value
```

Read it as:

> In rows where the condition is true, set this column to this value.

### Exercise B1: Flag high shipping fee

Create a column called `high_shipping_fee`.

Rules:

- default value: `0`
- set to `1` if `shipping_fee >= 10`

In [ ]:
# Exercise B1: Your code here

### Solution B1

In [ ]:
tx_all["high_shipping_fee"] = 0

tx_all.loc[
    tx_all["shipping_fee"] >= 10,
    "high_shipping_fee"
] = 1

tx_all[[
    "shipping_fee",
    "high_shipping_fee"
]].head(10)

### Exercise B2: Create `manual_order_segment`

Create a column called `manual_order_segment`.

Rules:

- default value: `"Normal"`
- set to `"Large Discount"` if `discount_rate >= 0.15`
- set to `"High Value"` if `net_sales >= 200`

Note: If a row satisfies both conditions, the second `.loc` update may overwrite the first one. This is a useful teaching point.

In [ ]:
# Exercise B2: Your code here

### Solution B2

In [ ]:
tx_all["manual_order_segment"] = "Normal"

tx_all.loc[
    tx_all["discount_rate"] >= 0.15,
    "manual_order_segment"
] = "Large Discount"

tx_all.loc[
    tx_all["net_sales"] >= 200,
    "manual_order_segment"
] = "High Value"

tx_all[[
    "net_sales",
    "discount_rate",
    "manual_order_segment"
]].head(15)

### Exercise B3: Use `.loc` with AND condition

Create `risky_return`.

Rules:

- default value: `0`
- set to `1` if:
  - `is_return == 1`
  - AND `net_sales >= 100`

In [ ]:
# Exercise B3: Your code here

### Solution B3

In [ ]:
tx_all["risky_return"] = 0

tx_all.loc[
    (tx_all["is_return"] == 1) & (tx_all["net_sales"] >= 100),
    "risky_return"
] = 1

tx_all[[
    "is_return",
    "net_sales",
    "risky_return"
]].head(15)

## C. More Practice: `np.where`

`np.where()` is useful for a simple two-way decision.

Pattern:

```python
np.where(condition, value_if_true, value_if_false)
```

It is like an if-else statement for a whole column.

### Exercise C1: Create `return_text`

Create a column called `return_text`.

Rules:

- `"Returned"` if `is_return == 1`
- `"Kept"` otherwise

In [ ]:
# Exercise C1: Your code here

### Solution C1

In [ ]:
tx_all["return_text"] = np.where(
    tx_all["is_return"] == 1,
    "Returned",
    "Kept"
)

tx_all[[
    "is_return",
    "return_text"
]].head(10)

### Exercise C2: Create `discount_text`

Create a column called `discount_text`.

Rules:

- `"Discounted"` if `discount_amount > 0`
- `"No Discount"` otherwise

In [ ]:
# Exercise C2: Your code here

### Solution C2

In [ ]:
tx_all["discount_text"] = np.where(
    tx_all["discount_amount"] > 0,
    "Discounted",
    "No Discount"
)

tx_all[[
    "discount_amount",
    "discount_text"
]].head(10)

### Exercise C3: Create `positive_revenue`

Create a column called `positive_revenue`.

Rules:

- `1` if `net_sales > 0`
- `0` otherwise

In [ ]:
# Exercise C3: Your code here

### Solution C3

In [ ]:
tx_all["positive_revenue"] = np.where(
    tx_all["net_sales"] > 0,
    1,
    0
)

tx_all[[
    "net_sales",
    "positive_revenue"
]].head()

### Exercise C4: Nested `np.where`

Create a column called `simple_sales_level`.

Rules:

- `"High"` if `net_sales >= 200`
- `"Medium"` if `net_sales >= 100`
- `"Low"` otherwise

Nested `np.where()` works, but for more than two conditions, `np.select()` is usually cleaner.

In [ ]:
# Exercise C4: Your code here

### Solution C4

In [ ]:
tx_all["simple_sales_level"] = np.where(
    tx_all["net_sales"] >= 200,
    "High",
    np.where(
        tx_all["net_sales"] >= 100,
        "Medium",
        "Low"
    )
)

tx_all[[
    "net_sales",
    "simple_sales_level"
]].head(10)

## D. More Practice: `np.select`

Use `np.select()` when you have multiple conditions.

The order matters because the first matching condition is used.

### Exercise D1: Create `order_size_v2`

Rules:

- `"Large"` if `net_sales >= 300`
- `"Medium"` if `net_sales` is between 100 and 299.99
- `"Small"` if `net_sales` is between 0.01 and 99.99
- `"No Revenue"` if `net_sales == 0`
- `"Unknown"` otherwise

In [ ]:
# Exercise D1: Your code here

### Solution D1

In [ ]:
conditions = [
    tx_all["net_sales"] >= 300,
    tx_all["net_sales"].between(100, 299.99),
    tx_all["net_sales"].between(0.01, 99.99),
    tx_all["net_sales"] == 0
]

choices = [
    "Large",
    "Medium",
    "Small",
    "No Revenue"
]

tx_all["order_size_v2"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

tx_all[[
    "net_sales",
    "order_size_v2"
]].head(15)

### Exercise D2: Create `customer_value_segment`

Rules:

- `"VIP High Order"` if `loyalty_tier` is `"Platinum"` and `net_sales >= 150`
- `"Loyal High Order"` if `loyalty_tier` is `"Gold"` and `net_sales >= 150`
- `"Regular High Order"` if `net_sales >= 150`
- `"Standard"` otherwise

In [ ]:
# Exercise D2: Your code here

### Solution D2

In [ ]:
conditions = [
    (tx_all["loyalty_tier"] == "Platinum") & (tx_all["net_sales"] >= 150),
    (tx_all["loyalty_tier"] == "Gold") & (tx_all["net_sales"] >= 150),
    tx_all["net_sales"] >= 150
]

choices = [
    "VIP High Order",
    "Loyal High Order",
    "Regular High Order"
]

tx_all["customer_value_segment"] = np.select(
    conditions,
    choices,
    default="Standard"
)

tx_all[[
    "loyalty_tier",
    "net_sales",
    "customer_value_segment"
]].head(15)

## E. More Practice: `lambda`

A `lambda` function is a short one-line function.

Example:

```python
lambda x: x * 2
```

means:

> take x and return x times 2.

In pandas, we often use lambda with:

- `.apply()`
- `.assign()`
- `.transform()`

### Exercise E1: Use lambda with one column

Create `net_sales_rounded`.

Round `net_sales` to 2 decimals using `.apply(lambda x: ...)`.

In [ ]:
# Exercise E1: Your code here

### Solution E1

In [ ]:
tx_all["net_sales_rounded"] = tx_all["net_sales"].apply(
    lambda x: round(x, 2)
)

tx_all[[
    "net_sales",
    "net_sales_rounded"
]].head()

### Exercise E2: Use lambda to create text

Create `sales_message`.

Rules:

- If `net_sales >= 150`, return `"Important order"`
- Otherwise return `"Normal order"`

In [ ]:
# Exercise E2: Your code here

### Solution E2

In [ ]:
tx_all["sales_message"] = tx_all["net_sales"].apply(
    lambda x: "Important order" if x >= 150 else "Normal order"
)

tx_all[[
    "net_sales",
    "sales_message"
]].head(10)

### Exercise E3: Lambda inside `.assign()`

Use `.assign()` and `lambda` to create:

`marketing_ratio = marketing_spend / net_sales`

To avoid division by zero, replace zero `net_sales` values with `np.nan`.

In [ ]:
# Exercise E3: Your code here

### Solution E3

In [ ]:
tx_all = tx_all.assign(
    marketing_ratio=lambda x: x["marketing_spend"] / x["net_sales"].replace(0, np.nan)
)

tx_all[[
    "marketing_spend",
    "net_sales",
    "marketing_ratio"
]].head()

## F. More Practice: `.apply()` with custom functions

Use a custom function when the business rule is easier to read as normal Python logic.

### Exercise F1: Create `order_size_function`

Write a function called `get_order_size`.

Rules:

- `"Large"` if `net_sales >= 300`
- `"Medium"` if `net_sales >= 100`
- `"Small"` if `net_sales > 0`
- `"No Revenue"` if `net_sales == 0`
- `"Unknown"` otherwise

Then apply it to `net_sales`.

In [ ]:
# Exercise F1: Your code here

### Solution F1

In [ ]:
def get_order_size(net_sales):
    if pd.isna(net_sales):
        return "Unknown"
    elif net_sales >= 300:
        return "Large"
    elif net_sales >= 100:
        return "Medium"
    elif net_sales > 0:
        return "Small"
    elif net_sales == 0:
        return "No Revenue"
    else:
        return "Unknown"

tx_all["order_size_function"] = tx_all["net_sales"].apply(get_order_size)

tx_all[[
    "net_sales",
    "order_size_function"
]].head(15)

### Exercise F2: Row-level apply with multiple columns

Create a function called `review_reason(row)`.

Rules:

- If `is_return == 1`, return `"Returned"`
- Else if `discount_rate >= 0.15`, return `"Large Discount"`
- Else if `net_sales >= 200`, return `"High Sales"`
- Otherwise return `"Normal"`

Then create a column called `review_reason`.

In [ ]:
# Exercise F2: Your code here

### Solution F2

In [ ]:
def review_reason(row):
    if row["is_return"] == 1:
        return "Returned"
    elif row["discount_rate"] >= 0.15:
        return "Large Discount"
    elif row["net_sales"] >= 200:
        return "High Sales"
    else:
        return "Normal"

tx_all["review_reason"] = tx_all.apply(
    review_reason,
    axis=1
)

tx_all[[
    "is_return",
    "discount_rate",
    "net_sales",
    "review_reason"
]].head(15)

### Exercise F3: Create `customer_message` using several columns

Write a function called `customer_message(row)`.

Rules:

- If `loyalty_tier == "Platinum"` and `net_sales >= 150`, return `"Thank VIP customer"`
- If `segment == "Student"` and `discount_amount > 0`, return `"Student discount order"`
- If `is_return == 1`, return `"Follow up on return"`
- Otherwise return `"Standard message"`

In [ ]:
# Exercise F3: Your code here

### Solution F3

In [ ]:
def customer_message(row):
    if (row["loyalty_tier"] == "Platinum") and (row["net_sales"] >= 150):
        return "Thank VIP customer"
    elif (row["segment"] == "Student") and (row["discount_amount"] > 0):
        return "Student discount order"
    elif row["is_return"] == 1:
        return "Follow up on return"
    else:
        return "Standard message"

tx_all["customer_message"] = tx_all.apply(
    customer_message,
    axis=1
)

tx_all[[
    "loyalty_tier",
    "segment",
    "discount_amount",
    "is_return",
    "net_sales",
    "customer_message"
]].head(15)

## G. More Practice: Group-Based Column Creation with `transform()`

Use `transform()` when each row needs information about its group.

Common examples:

- group average
- group total
- group maximum
- group minimum
- group percentile
- group count

The key idea:

> `transform()` returns a result with the same number of rows as the original DataFrame.

### Exercise G1: Compare each order with channel average

Create:

- `channel_avg_net_sales`
- `above_channel_average_v2`

The second column should be 1 if the order's `net_sales` is above the average for its channel.

In [ ]:
# Exercise G1: Your code here

### Solution G1

In [ ]:
tx_all["channel_avg_net_sales"] = (
    tx_all
    .groupby("channel")["net_sales"]
    .transform("mean")
)

tx_all["above_channel_average_v2"] = (
    tx_all["net_sales"] > tx_all["channel_avg_net_sales"]
).astype(int)

tx_all[[
    "channel",
    "net_sales",
    "channel_avg_net_sales",
    "above_channel_average_v2"
]].head(10)

### Exercise G2: Category maximum

Create:

- `category_max_sales`
- `pct_of_category_max`

Formula:

`pct_of_category_max = net_sales / category_max_sales`

In [ ]:
# Exercise G2: Your code here

### Solution G2

In [ ]:
tx_all["category_max_sales"] = (
    tx_all
    .groupby("category")["net_sales"]
    .transform("max")
)

tx_all["pct_of_category_max"] = (
    tx_all["net_sales"] / tx_all["category_max_sales"]
)

tx_all[[
    "category",
    "net_sales",
    "category_max_sales",
    "pct_of_category_max"
]].head(10)

### Exercise G3: Customer average discount

Create:

- `customer_avg_discount_rate`
- `above_customer_avg_discount`

The second column should be 1 if the order's discount rate is above that customer's average discount rate.

In [ ]:
# Exercise G3: Your code here

### Solution G3

In [ ]:
tx_all["customer_avg_discount_rate"] = (
    tx_all
    .groupby("customer_id")["discount_rate"]
    .transform("mean")
)

tx_all["above_customer_avg_discount"] = (
    tx_all["discount_rate"] > tx_all["customer_avg_discount_rate"]
).astype(int)

tx_all[[
    "customer_id",
    "discount_rate",
    "customer_avg_discount_rate",
    "above_customer_avg_discount"
]].head(15)

### Exercise G4: Count transactions per customer

Create a column called `customer_transaction_count`.

It should show how many transactions each customer has.

In [ ]:
# Exercise G4: Your code here

### Solution G4

In [ ]:
tx_all["customer_transaction_count"] = (
    tx_all
    .groupby("customer_id")["order_id"]
    .transform("count")
)

tx_all[[
    "customer_id",
    "order_id",
    "customer_transaction_count"
]].head(15)

## H. Mini Case Study: Build a Customer Risk Score

Now combine several concepts.

We will create a simple `risk_score`.

Add 1 point for each condition:

- order is returned
- discount rate is at least 15%
- net sales is below the customer's average sales
- shipping fee is at least 10

This is not a real risk model. It is a teaching example.

In [ ]:
# Step 1: Customer average net sales
tx_all["customer_avg_net_sales"] = (
    tx_all
    .groupby("customer_id")["net_sales"]
    .transform("mean")
)

# Step 2: Create the risk score
tx_all["risk_score"] = (
    (tx_all["is_return"] == 1).astype(int)
    + (tx_all["discount_rate"] >= 0.15).astype(int)
    + (tx_all["net_sales"] < tx_all["customer_avg_net_sales"]).astype(int)
    + (tx_all["shipping_fee"] >= 10).astype(int)
)

tx_all[[
    "customer_id",
    "net_sales",
    "customer_avg_net_sales",
    "is_return",
    "discount_rate",
    "shipping_fee",
    "risk_score"
]].head(15)

### Exercise H1

Create a column called `risk_level`.

Rules:

- `"High Risk"` if `risk_score >= 3`
- `"Medium Risk"` if `risk_score == 2`
- `"Low Risk"` otherwise

Use `np.select()`.

In [ ]:
# Exercise H1: Your code here

### Solution H1

In [ ]:
conditions = [
    tx_all["risk_score"] >= 3,
    tx_all["risk_score"] == 2
]

choices = [
    "High Risk",
    "Medium Risk"
]

tx_all["risk_level"] = np.select(
    conditions,
    choices,
    default="Low Risk"
)

tx_all[[
    "risk_score",
    "risk_level"
]].head(15)

## I. More Practice: Dummy Variables

Dummy variables convert categories into numeric columns.

This is often needed for machine learning or regression.

### Exercise I1: Create dummies for `segment`

Create dummy columns for `segment` with prefix `"segment"`.

Then combine them with `tx_all`.

In [ ]:
# Exercise I1: Your code here

### Solution I1

In [ ]:
segment_dummies = pd.get_dummies(
    tx_all["segment"],
    prefix="segment",
    dtype=int
)

tx_all = pd.concat(
    [tx_all, segment_dummies],
    axis=1
)

tx_all.head()

### Exercise I2: Create dummies and drop the first category

Create dummy columns for `channel` using `drop_first=True`.

This is common in regression to avoid perfect multicollinearity.

In [ ]:
# Exercise I2: Your code here

### Solution I2

In [ ]:
channel_dummies_drop_first = pd.get_dummies(
    tx_all["channel"],
    prefix="channel_reg",
    dtype=int,
    drop_first=True
)

channel_dummies_drop_first.head()

## J. Debugging Exercises

These are useful for classroom discussion.

### Debug Exercise J1: What is wrong with this code?

```python
tx_all[tx_all["net_sales"] > 100]["flag"] = 1
```

Answer:

This is chained assignment. It may update a copy, not the original DataFrame.

Correct version:

In [ ]:
tx_all["flag"] = 0

tx_all.loc[
    tx_all["net_sales"] > 100,
    "flag"
] = 1

tx_all[[
    "net_sales",
    "flag"
]].head()

### Debug Exercise J2: What is wrong with this code?

```python
tx_all["net_sales"] > 100 & tx_all["is_return"] == 1
```

Answer:

Each condition must be inside parentheses.

Correct version:

In [ ]:
condition = (
    (tx_all["net_sales"] > 100)
    & (tx_all["is_return"] == 1)
)

condition.head()

## K. Additional Final Practice Questions

Use these as in-class activities, homework, or quiz practice.

### Exercise K1

Create `free_shipping_flag`: 1 if `shipping_fee == 0`, otherwise 0.

In [ ]:
# Exercise K1: Your code here

### Solution K1

In [ ]:
tx_all["free_shipping_flag"] = (tx_all["shipping_fee"] == 0).astype(int)

tx_all[["shipping_fee", "free_shipping_flag"]].head()

### Exercise K2

Create `big_discount_or_return`: 1 if `discount_rate >= 0.20` OR `is_return == 1`, otherwise 0.

In [ ]:
# Exercise K2: Your code here

### Solution K2

In [ ]:
tx_all["big_discount_or_return"] = (
    (tx_all["discount_rate"] >= 0.20)
    | (tx_all["is_return"] == 1)
).astype(int)

tx_all[["discount_rate", "is_return", "big_discount_or_return"]].head()

### Exercise K3

Create `category_order_count`: number of orders in each category.

In [ ]:
# Exercise K3: Your code here

### Solution K3

In [ ]:
tx_all["category_order_count"] = (
    tx_all
    .groupby("category")["order_id"]
    .transform("count")
)

tx_all[["category", "order_id", "category_order_count"]].head()

### Exercise K4

Create `country_return_rate`: average return rate for each country.

In [ ]:
# Exercise K4: Your code here

### Solution K4

In [ ]:
tx_all["country_return_rate"] = (
    tx_all
    .groupby("country")["is_return"]
    .transform("mean")
)

tx_all[["country", "is_return", "country_return_rate"]].head()

### Exercise K5

Create `above_country_return_rate`: 1 if the row is a return and the country return rate is above 0.10.

In [ ]:
# Exercise K5: Your code here

### Solution K5

In [ ]:
tx_all["above_country_return_rate"] = (
    (tx_all["is_return"] == 1)
    & (tx_all["country_return_rate"] > 0.10)
).astype(int)

tx_all[["country", "is_return", "country_return_rate", "above_country_return_rate"]].head()

## Expanded Module Summary

You now have multiple ways to create and update columns:

| Tool | Best for |
|---|---|
| direct math | simple numeric columns |
| `.loc` | updating selected rows |
| `np.where()` | simple if-else logic |
| `np.select()` | multiple conditions |
| `lambda` | short one-line functions |
| `.apply()` | row-level custom logic |
| `def` functions | readable complex logic |
| `.assign()` | pipeline-style column creation |
| `groupby().transform()` | subgroup-based row-level columns |
| `pd.get_dummies()` | categorical variables into 0/1 columns |

For beginners, the most important tools are:

1. Direct column creation
2. `.loc`
3. `np.where`
4. `groupby().transform()`

These four will solve many real business data problems.